In [2]:
import os
import operator
from typing import TypedDict, Annotated, Optional, get_type_hints
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
# reducer



In [4]:
# state -> dict
# class State(TypedDict):
#     msges : list

# def node_a(State): return {'msges' : ['A'] + 'aaa'}  # 1
# def node_b(State): return {'msges' : ['b']}  # 2

In [5]:
from typing import get_type_hints
def my_reducer(cur, new):
    return (cur or []) + new

class AnnotatedState(TypedDict):
    items : Annotated[list, my_reducer]

hints = get_type_hints(AnnotatedState, include_extras=True)
hints

{'items': typing.Annotated[list, <function my_reducer at 0x7f4fe24e16c0>]}

In [6]:
# items: Annotated[list, '아이템을 담는 변수입니다']

In [7]:
operator.add([1], [2]), operator.add(1, 2)

([1, 2], 3)

In [8]:
[1] + [2]

[1, 2]

In [9]:
# parser | retr | | | |

In [10]:
import operator
class AccumState(TypedDict):
    msgs : Annotated[list, operator.add]

def node_a(state): return {'msgs' : ['A 추가']}
def node_b(state): return {'msgs' : ['B 추가']}

b = StateGraph(AccumState)
b.add_node('node_a', node_a)
b.add_node('node_b', node_b)

b.add_edge(START, 'node_a')
b.add_edge('node_a', 'node_b')
b.add_edge('node_b', END)
app = b.compile()
app.invoke({'msgs' : []})

{'msgs': ['A 추가', 'B 추가']}

In [11]:
operator.add

<function _operator.add(a, b, /)>

In [12]:
ScoreState : scores list
    

add_kor : scores : 80

add_eng : scores : 90

add_math : scores : 70

SyntaxError: invalid syntax (1995194093.py, line 1)

In [13]:
class ScoreState(TypedDict):
    scores : Annotated[list, operator.add]

def add_kor(state): return {'scores' : [80]}
def add_eng(state): return {'scores' : [90]}
def add_math(state): return {'scores' : [70]}

b = StateGraph(ScoreState)
b.add_node('add_kor', add_kor)
b.add_node('add_eng', add_eng)
b.add_node('add_math', add_math)

b.add_edge(START, 'add_kor')
b.add_edge('add_kor', 'add_eng')
b.add_edge('add_eng', 'add_math')
b.add_edge('add_math', END)
app = b.compile()
app.invoke({'scores' : []})

{'scores': [80, 90, 70]}

In [14]:
from langgraph.graph.message import add_messages

In [15]:
# operator.add   add_messages
class ChatState(TypedDict):
    messages : Annotated[list, add_messages]
        
# def system_says(state):
#     return {'message' : [System~~]}

def user_says(state):
    return {'messages' : [HumanMessage(content='안녕')]}

def bot_says(state):
    return {'messages' : [AIMessage(content='반갑습니다')]}

b = StateGraph(ChatState)
b.add_node('user_says', user_says)
b.add_node('bot_says', bot_says)

b.add_edge(START, 'user_says')
b.add_edge('user_says', 'bot_says')
b.add_edge('bot_says', END)
app = b.compile()

result = app.invoke({'message' : []})
result

{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='74e304e6-3b4d-465f-8324-e527faacaf75'),
  AIMessage(content='반갑습니다', additional_kwargs={}, response_metadata={}, id='3e33f260-9967-49d2-85d3-e27630d41767', tool_calls=[], invalid_tool_calls=[])]}

In [16]:
sys_m = SystemMessage(content ='당신은 컴퓨터 전문가입니다')
hum_m = HumanMessage(content = '랭그래프?')
ai_m = AIMessage(content = '그래프 기반 라이브러리')
tool_m = ToolMessage(content = '검색결과....', tool_call_id= 'call_abc123')

In [17]:
for m in [sys_m, hum_m, ai_m, tool_m]:
    print(f"type = {m.type} | class = {type(m).__name__} | content = {m.content}")

type = system | class = SystemMessage | content = 당신은 컴퓨터 전문가입니다
type = human | class = HumanMessage | content = 랭그래프?
type = ai | class = AIMessage | content = 그래프 기반 라이브러리
type = tool | class = ToolMessage | content = 검색결과....


In [18]:
add_messages(add_messages(sys_m, hum_m), ai_m)

[SystemMessage(content='당신은 컴퓨터 전문가입니다', additional_kwargs={}, response_metadata={}, id='4ea9ea39-4736-4b37-bd19-49235e46c3be'),
 HumanMessage(content='랭그래프?', additional_kwargs={}, response_metadata={}, id='30d694b8-39e4-4867-b8c6-e53b32244d83'),
 AIMessage(content='그래프 기반 라이브러리', additional_kwargs={}, response_metadata={}, id='e4b6fbca-6578-47b0-abc8-16f2c3786e10', tool_calls=[], invalid_tool_calls=[])]

In [19]:
ai_with_tools = AIMessage(
    content = "",
    tool_calls =[
        {'name' : 'search', 'args' : {'q' : '랭그래프'}, 'id' : 'call_001'},
        {'name' : 'calc', 'args' : {'x' : 7, 'y':3}, 'id' : 'call_002'}
    ]
)

In [20]:
ai_with_tools

AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'search', 'args': {'q': '랭그래프'}, 'id': 'call_001', 'type': 'tool_call'}, {'name': 'calc', 'args': {'x': 7, 'y': 3}, 'id': 'call_002', 'type': 'tool_call'}], invalid_tool_calls=[])

In [21]:
ai_with_tools.content

''

In [22]:
len(ai_with_tools.tool_calls)

2

In [23]:
for tc in ai_with_tools.tool_calls:
    print(f" - {tc['name']} ({tc['args']}), id = {tc['id']}")

 - search ({'q': '랭그래프'}), id = call_001
 - calc ({'x': 7, 'y': 3}), id = call_002


In [24]:
m1 = AIMessage(content = '첫 응답', id = 'msg_x')
m2 = AIMessage(content = '두번째 응답', id = 'msg_x')

merged = add_messages([m1], [m2])
merged

[AIMessage(content='두번째 응답', additional_kwargs={}, response_metadata={}, id='msg_x', tool_calls=[], invalid_tool_calls=[])]

In [25]:
operator.add([1],[2])

[1, 2]

In [26]:
# ReAct : 
# 25 : 250 5

In [27]:
# def my_reducer(cur, new):
#     return (cur or []) + new

In [28]:
def keep_max(cur, new):   # 10  30
    return max(cur or 0, new)  # 30

class MaxState(TypedDict):
    score : Annotated[int, keep_max]
        
def round1(state) : return {'score' : 50}
def round2(state) : return {'score' : 80}
def round3(state) : return {'score' : 70}

b = StateGraph(MaxState)
for n, fn in [('round1', round1), ('round2', round2), ('round3', round3)]:
    b.add_node(n, fn)
b.add_edge(START, 'round1')
b.add_edge('round1', 'round2')
b.add_edge('round2', 'round3')
b.add_edge('round3', END)
app = b.compile()

app.invoke({'score' : 0})

{'score': 80}

In [29]:
def add_int(cur, new) :
    return (cur or 0) + new

class SumState(TypedDict):
    total : Annotated[int, add_int]

def s1(state) : return {'total' : 10}
def s2(state) : return {'total' : 20}
def s3(state) : return {'total' : 30}

b = StateGraph(SumState)
for n, fn in [('s1', s1), ('s2', s2), ('s3', s3)]:
    b.add_node(n, fn)
b.add_edge(START, 's1')
b.add_edge('s1', 's2')
b.add_edge('s2', 's3')
b.add_edge('s3', END)
app = b.compile()

app.invoke({'total' : 0})

{'total': 60}

In [30]:
def merge_dicts(cur, new):
    return {**(cur or {}), **(new or {})}

class MetaState(TypedDict):
    meta : Annotated[dict, merge_dicts]

def fetch_user(state) : 
    #####
    return {'meta' : {'user_id':'u123'}}
def fetch_session(state) : return {'meta' : {'session_id':'s456'}}
def fetch_locale(state) : return {'meta' : {'locale':'ko-KR'}}

b = StateGraph(MetaState)
for n, fn in [('fetch_user', fetch_user), ('fetch_session', fetch_session), ('fetch_locale', fetch_locale)]:
    b.add_node(n, fn)
    
b.add_edge(START, 'fetch_user')
b.add_edge('fetch_user', 'fetch_session')
b.add_edge('fetch_session', 'fetch_locale')
b.add_edge('fetch_locale', END)
app = b.compile()

app.invoke({'meta' : {}})

{'meta': {'user_id': 'u123', 'session_id': 's456', 'locale': 'ko-KR'}}

In [32]:
def merge_dicts(cur, new):
    return {**(cur or {}), **(new or {})}

ConfigState(TypedDict):
    config(dict)

set_model        model         gpt-4o-mini
set_temperature  temperature   2.0
override_model   model         gpt-4o

SyntaxError: invalid decimal literal (1546610739.py, line 7)

In [ ]:
class ConfigState(TypedDict):
    config: Annotated[dict, merge_dicts]


def set_model(state):
    return {"config": {"model": "gpt-4o-mini"}}


def set_temperature(state):
    return {"config": {"temperature": 0.5}}


def set_override_model(state):
    return {"config": {"model": "gpt-4o"}}


builder = StateGraph(ConfigState)
builder.add_node("set_model", set_model)
builder.add_node("set_temperature", set_temperature)
builder.add_node("set_override_model", set_override_model)

builder.add_edge(START, "set_model")
builder.add_edge("set_model", "set_temperature")
builder.add_edge("set_temperature", "set_override_model")
builder.add_edge("set_override_model", END)

app = builder.compile()
app.invoke({"config": {}})


In [ ]:
class MultiState(TypedDict):
    messages : Annotated[list, add_messages]
    score : Annotated[int, keep_max]
    meta : Annotated[dict, merge_dicts]
    raw : str

def n1(state):
    return {'messages' : [HumanMessage(content='안녕')],
           'score' : 50, 'meta' : {"step" : 1}, 'raw' : 'first'}
def n2(state):
    return {'messages' : [AIMessage(content='반가워')],
           'score' : 80, 'meta' : {'step': 2, 'ok' : True}, 'raw' : 'second'}

b = StateGraph(MultiState)
b.add_node('n1', n1)
b.add_node('n2', n2)
b.add_edge(START, 'n1')
b.add_edge('n1', 'n2')
b.add_edge('n2', END)
app = b.compile()

result = app.invoke({'messages' : [], 'score' : 0, 'meta' : {}, 'raw' : ''})

In [ ]:
result

In [ ]:
len(result['messages'])

In [ ]:
GameState
events : list -> 이벤트 로그를 출력
best : int -> 최고 점수
stats : dict => merge_dicts
    
r1 : events = ['round1'], best = 30, stats = {'plays': 1}
r2 : events = ['round2'], best = 50, stats = {'wins': 1}

In [33]:
class GameState(TypedDict):
    events: Annotated[list, operator.add]
    best: Annotated[int, keep_max]
    stats: Annotated[dict, merge_dicts]

def play_round1(state):
    return {"events": ["round1"], "best": 30, "stats": {"plays": 1}}
def play_round2(state):
    return {"events": ["round2"], "best": 50, "stats": {"wins": 1}}

b = StateGraph(GameState)
b.add_node("play_round1", play_round1)
b.add_node("play_round2", play_round2)
b.add_edge(START, "play_round1")
b.add_edge("play_round1", "play_round2")
b.add_edge("play_round2", END)
app = b.compile()

print(app.invoke({"events": [], "best": 0, "stats": {}}))

{'events': ['round1', 'round2'], 'best': 50, 'stats': {'plays': 1, 'wins': 1}}


In [35]:
class TrimState(TypedDict):
    messages : Annotated[list, add_messages]
    visible : list

N = 3

def add_many(state):
    new_msgs = [HumanMessage(content =f"msg-{i}") for i in range(10)]
    return {'messages' : new_msgs}
def trim_node(state):
    return {'visible' : state['messages'][-N:]}

b= StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)
app = b.compile()

result = app.invoke({'messages' : [], 'visible' : []})

In [36]:
result

{'messages': [HumanMessage(content='msg-0', additional_kwargs={}, response_metadata={}, id='470065bf-ad75-4293-be10-28dfeb219b09'),
  HumanMessage(content='msg-1', additional_kwargs={}, response_metadata={}, id='72d52062-dbcd-4163-9289-46d134f4ea31'),
  HumanMessage(content='msg-2', additional_kwargs={}, response_metadata={}, id='5f2d8bb0-382a-4d1a-b487-011bf5eb23eb'),
  HumanMessage(content='msg-3', additional_kwargs={}, response_metadata={}, id='ac6323a5-b0ab-4234-bb8c-22bc97be9521'),
  HumanMessage(content='msg-4', additional_kwargs={}, response_metadata={}, id='ff1c5904-2b06-4af2-b6d2-623cf50e1140'),
  HumanMessage(content='msg-5', additional_kwargs={}, response_metadata={}, id='8fa0c78b-0d95-4317-bf52-1bce081a3cc2'),
  HumanMessage(content='msg-6', additional_kwargs={}, response_metadata={}, id='53c0b6f0-2015-41df-9a3c-7c05be794451'),
  HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='675dcda5-7f41-4c37-8cd1-ed808970bac5'),
  HumanMessage(content='msg-

In [37]:
from langgraph.graph.message import RemoveMessage

In [38]:
[1,2,3,4,5,6,7,8,9,10][-3:]

[8, 9, 10]

In [39]:
[1,2,3,4,5,6,7,8,9,10][:-3]

[1, 2, 3, 4, 5, 6, 7]

In [40]:
class TrimState(TypedDict):
    messages : Annotated[list, add_messages]

N = 3

def add_many(state):
    new_msgs = [HumanMessage(content =f"msg-{i}") for i in range(10)]
    return {'messages' : new_msgs}

def trim_node(state):
    msgs = state['messages']
    to_remove = msgs[:-N]
    return {'messages' : [RemoveMessage(id=m.id) for m in to_remove]}

b= StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)
app = b.compile()

result = app.invoke({'messages' : []})

In [41]:
result

{'messages': [HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='f2a7c333-8835-41f1-bf80-2d18e69e8373'),
  HumanMessage(content='msg-8', additional_kwargs={}, response_metadata={}, id='b8930a9b-57ac-4a97-9fb1-bced1080105d'),
  HumanMessage(content='msg-9', additional_kwargs={}, response_metadata={}, id='081c0dd4-1ebd-448d-a042-d9cdfa4dc0c8')]}

In [43]:
mixed = [
    SystemMessage(content = '시스템'),
    HumanMessage(content ='안녕'),
    AIMessage(content = '반가워'),
    ToolMessage(content = '검색 결과', tool_call_id= 't1'),
    HumanMessage(content = 'LangGraph')
]

def only_human(msgs):
    return [m for m in msgs if isinstance(m, HumanMessage)]

def drop_tool(msgs):
    return [m for m in msgs if not isinstance(m, ToolMessage)]

drop_tool(mixed)

[SystemMessage(content='시스템', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반가워', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='LangGraph', additional_kwargs={}, response_metadata={})]

In [ ]:
FilterState : messages (list), reply (str)
    
seed : return {'messages' :     [SystemMessage(content = '시스템'),
    HumanMessage(content ='안녕'),
    AIMessage(content = '반가워'),
    ToolMessage(content = '검색 결과', tool_call_id= 't1'),
    HumanMessage(content = 'LangGraph')]}

respond :
    tool message drop
    response = llm.invoke(dropped message)
    return {'reply' : response.content}


In [44]:
class FilterState(TypedDict):
    messages: Annotated[list, add_messages]
    reply: str

def seed(state):
    return {"messages": [
        SystemMessage(content="당신은 친절한 도우미"),
        HumanMessage(content="LangGraph는 무엇인가요?"),
        AIMessage(content="확인해보겠습니다."),
        ToolMessage(content="(검색 결과 raw)", tool_call_id="tx"),
    ]}

def respond(state):
    filtered = drop_tool(state["messages"])
    response = llm.invoke(filtered)
    return {"reply": response.content}

b = StateGraph(FilterState)
b.add_node("seed", seed)
b.add_node("respond", respond)

b.add_edge(START, "seed")
b.add_edge("seed", "respond")
b.add_edge("respond", END)
app = b.compile()

print(app.invoke({"messages": [], "reply": ""})["reply"][:150])

LangGraph는 언어와 그래프 데이터 구조를 결합하여 다양한 자연어 처리(NLP) 작업을 수행할 수 있는 플랫폼이나 도구로 추측됩니다. 이는 언어 모델의 기능을 강화하거나, 텍스트와 관련된 정보를 더 구조화된 방식으로 시각화 및 분석하는 데 사용될 수 있습니다.




In [45]:
class TurnState(TypedDict):
    messages : Annotated[list, add_messages]

def chat_turn(state):
    response = llm.invoke(state["messages"])
    return {'messages' : [response]}

b = StateGraph(TurnState)
b.add_node('chat_turn', chat_turn)
b.add_edge(START, 'chat_turn')
b.add_edge('chat_turn', END)
turn_app = b.compile()

history = [SystemMessage(content ='당신은 짧게 대답하는 도우미입니다')]
for user_text in ['안녕!', '내 이름은 이상호야', '내 이름이 뭐였지?']:
    out = turn_app.invoke({'messages' : history + [HumanMessage(content=user_text)]})
    history = out['messages']
    print(f"USER: {user_text}")
    print(f"BOT: {history[-1].content}")

USER: 안녕!
BOT: 안녕하세요! 어떻게 도와드릴까요?
USER: 내 이름은 이상호야
BOT: 만나서 반가워요, 이상호님! 무엇을 도와드릴까요?
USER: 내 이름이 뭐였지?
BOT: 이상호님이신데요.


In [ ]:
llm.invoke()